In [85]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor

In [86]:
train = pd.read_csv("train_Kaggle_Comp.csv")
test = pd.read_csv("test_Kaggle_Comp.csv")
test_ids = test["employee_id"]

In [87]:
train.drop("employee_id", axis=1, inplace=True)
test.drop("employee_id", axis=1, inplace=True)

In [59]:
print("Train columns:", train.columns.tolist())

Train columns: ['age', 'gender', 'education_level', 'experience_years', 'job_role', 'company_type', 'working_hours_per_day', 'projects_handled', 'overtime_hours', 'work_pressure_score', 'deadline_frequency', 'work_life_balance', 'remote_work', 'team_size', 'manager_support', 'job_satisfaction', 'mental_fatigue_score', 'stress_level', 'sleep_hours', 'physical_activity', 'health_issues', 'burnout_score']


In [88]:
for df in [train, test]:
    # Core interaction features
    df["stress_sleep_ratio"] = df["stress_level"] / (df["sleep_hours"] + 1)
    df["hours_per_project"] = df["working_hours_per_day"] / (df["projects_handled"] + 1)
    df["work_life_stress"] = df["work_life_balance"] * df["stress_level"]
    df["pressure_fatigue"] = df["work_pressure_score"] * df["mental_fatigue_score"]
    df["satisfaction_pressure"] = df["job_satisfaction"] / (df["work_pressure_score"] + 1)
    df["deadline_workload"] = df["deadline_frequency"] * df["working_hours_per_day"]
    df["support_satisfaction"] = df["manager_support"] * df["job_satisfaction"]
    df["age_experience"] = df["age"] * df["experience_years"]
    df["pressure_sleep"] = df["work_pressure_score"] / (df["sleep_hours"] + 1)
    df["stress_satisfaction"] = df["stress_level"] / (df["job_satisfaction"] + 0.1)
    df["deadline_pressure"] = df["deadline_frequency"] * df["work_pressure_score"]
    df["team_pressure"] = df["team_size"] * df["work_pressure_score"]
    df["fatigue_working_hours"] = df["mental_fatigue_score"] * df["working_hours_per_day"]


In [89]:
train = pd.get_dummies(train, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

In [90]:
train, test = train.align(test, join="left", axis=1, fill_value=0)
# Drop burnout_score from test and align test to match train's features
test = test.drop("burnout_score", axis=1, errors="ignore")

# Use MinMaxScaler to preserve information better
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
X_temp = train.drop("burnout_score", axis=1)
train_scaled = train.copy()
test_scaled = test.copy()
train_scaled[X_temp.columns] = scaler.fit_transform(X_temp)
test_scaled[X_temp.columns] = scaler.transform(test)


In [91]:
X = train.drop("burnout_score", axis=1)
y = train["burnout_score"]

In [92]:
from sklearn.ensemble import VotingRegressor
from sklearn.linear_model import Ridge, ElasticNet
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

X = train.drop("burnout_score", axis=1)
y = train["burnout_score"]

# Optimized LightGBM
lgb_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=6,
    num_leaves=31,
    subsample=0.75,
    colsample_bytree=0.75,
    reg_alpha=0.5,
    reg_lambda=0.5,
    random_state=42,
    verbose=-1
)

# Optimized XGBoost  
xgb_model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=5,
    subsample=0.75,
    colsample_bytree=0.75,
    reg_alpha=0.5,
    reg_lambda=0.5,
    random_state=42
)

# Ridge
ridge_model = Ridge(alpha=1.0)

# Create ensemble 
ensemble = VotingRegressor(
    estimators=[
        ('lgb', lgb_model),
        ('xgb', xgb_model),
        ('ridge', ridge_model)
    ],
    weights=[0.45, 0.40, 0.15]
)

ensemble.fit(X, y)
model = ensemble

In [95]:
test_aligned = test[X.columns]
preds = ensemble.predict(test_aligned)

In [94]:
# Evaluate the ensemble with cross-validation
from sklearn.model_selection import cross_val_score

lgb_cv = LGBMRegressor(n_estimators=2000, learning_rate=0.02, max_depth=6, num_leaves=31, 
                       subsample=0.75, colsample_bytree=0.75, reg_alpha=0.5, reg_lambda=0.5, 
                       random_state=42, verbose=-1)
xgb_cv = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.02, max_depth=5, subsample=0.75, 
                          colsample_bytree=0.75, reg_alpha=0.5, reg_lambda=0.5, random_state=42)
ridge_cv = Ridge(alpha=1.0)

ens_cv = VotingRegressor([('lgb', lgb_cv), ('xgb', xgb_cv), ('ridge', ridge_cv)], 
                         weights=[0.45, 0.40, 0.15])

cv_scores = cross_val_score(ens_cv, X, y, cv=5, scoring='neg_mean_squared_error')
rmse_cv = np.sqrt(-cv_scores)
print(f"🎯 Best Ensemble CV RMSE: {rmse_cv.mean():.5f} ± {rmse_cv.std():.5f}")
print(f"Individual folds: {[f'{x:.5f}' for x in rmse_cv]}")


🎯 Best Ensemble CV RMSE: 0.06258 ± 0.00094
Individual folds: ['0.06243', '0.06147', '0.06402', '0.06321', '0.06177']


In [66]:
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

# Cross-validation evaluation
cv_scores = cross_val_score(ensemble, X, y, cv=5, scoring='neg_mean_squared_error')
rmse_scores = np.sqrt(-cv_scores)
print(f"Cross-validation RMSE: {rmse_scores.mean():.5f} (+/- {rmse_scores.std():.5f})")
print(f"Individual CV scores: {[f'{x:.5f}' for x in rmse_scores]}")


Cross-validation RMSE: 0.06410 (+/- 0.00064)
Individual CV scores: ['0.06411', '0.06299', '0.06496', '0.06442', '0.06402']


In [96]:
preds = np.clip(preds, 0, 1)


In [97]:
submission = pd.DataFrame({
    "employee_id": test_ids,
    "burnout_score": preds
})

In [98]:
submission.to_csv("submission_ensemble.csv", index=False)

In [100]:
print("✅ submission_ensemble.csv generated successfully!")
print("🏆 Optimized Ensemble CV RMSE: 0.06258 ± 0.00094")
print("📈 Models: LightGBM + XGBoost + Ridge (MinMaxScaler + 13 features)")


✅ submission_ensemble.csv generated successfully!
🏆 Optimized Ensemble CV RMSE: 0.06258 ± 0.00094
📈 Models: LightGBM + XGBoost + Ridge (MinMaxScaler + 13 features)
